In [1]:
import torch
from einops import rearrange, einsum

In [2]:
images = torch.randn(64, 128, 128, 3)
dim_by = torch.linspace(start=0.0, end=1.0, steps=10)
dim_value = rearrange(dim_by, "dim_value-> 1 dim_value 1 1 1")

In [3]:
dim_value.shape

torch.Size([1, 10, 1, 1, 1])

In [4]:
images_rearr = rearrange(images, "b height width channel -> b 1 height width channel")
images_rearr.shape

torch.Size([64, 1, 128, 128, 3])

In [5]:
images = torch.randn(64, 128, 128, 3)  # (batch, height, width, channel)
dim_by = torch.linspace(start=0.0, end=1.0, steps=10)
## Reshape and multiply
dim_value = rearrange(dim_by, "dim_value -> 1 dim_value 1 1 1")
images_rearr = rearrange(images, "b height width channel -> b 1 height width channel")
dimmed_images = images_rearr * dim_value
## Or in one go:
dimmed_images = einsum(
    images,
    dim_by,
    "batch height width channel, dim_value -> batch dim_value height width channel",
)

In [6]:
dimmed_images.shape

torch.Size([64, 10, 128, 128, 3])

In [7]:
import einx


In [8]:
channels_last = torch.randn(64, 32, 32, 3)
B = torch.randn(32 * 32, 32 * 32)
# (batch, height, width, channel)
## Rearrange an image tensor for mixing across all pixels
channels_last_flat = channels_last.view(
    -1, channels_last.size(1) * channels_last.size(2), channels_last.size(3)
)
channels_last_flat.shape

torch.Size([64, 1024, 3])

In [9]:
channels_first_flat = channels_last_flat.transpose(1, 2)
channels_first_flat.shape

torch.Size([64, 3, 1024])

In [10]:
channels_first_flat_transformed = channels_first_flat @ B.T
channels_first_flat_transformed.shape

torch.Size([64, 3, 1024])

In [11]:
channels_last_flat_transformed = channels_first_flat_transformed.transpose(1, 2)
channels_last_flat_transformed.shape

torch.Size([64, 1024, 3])

In [13]:
channels_last_transformed = channels_last_flat_transformed.view(*channels_last.shape)
channels_last_transformed.shape

torch.Size([64, 32, 32, 3])

In [14]:
# Instead, using einops:
height = width = 32
## Rearrange replaces clunky torch view + transpose
channels_first = rearrange(
    channels_last, "batch height width channel -> batch channel (height width)"
)
channels_first_transformed = einsum(
    channels_first,
    B,
    "batch channel pixel_in, pixel_out pixel_in -> batch channel pixel_out",
)
channels_last_transformed = rearrange(
    channels_first_transformed,
    "batch channel (height width) -> batch height width channel",
    height=height,
    width=width,
)
# Or, if you’re feeling crazy: all in one go using einx.dot (einx equivalent of einops.einsum)
height = width = 32
channels_last_transformed = einx.dot(
    "batch row_in col_in channel, (row_out col_out) (row_in col_in)"
    "-> batch row_out col_out channel",
    channels_last,
    B,
    col_in=width,
    col_out=width,
)